# Lezione 2: L'Integratore ODE (Test-Time Inference)
**File**: [`cfm/solver.py`](file:///g:/My%20Drive/PhD%20&%20Research/Temporal%20GNN/cfm_grn/cfm/solver.py)  
**Ruolo nel progetto**: Trasforma il campo vettoriale appreso $v_\theta$ in previsioni biologiche concrete integrando la traiettoria da $t=0$ a $t=1$.


---
## 1. Dalla Velocità alla Traiettoria: La Teoria dell'Integrazione

Durante l'addestramento (Lezione 1), abbiamo insegnato alla rete neurale a predire la velocità istantanea di traslazione:
$$\frac{dx_t}{dt} = v_\theta(x_t, t, c)$$

Al momento dell'inferenza (test-time), disponiamo di una popolazione di cellule sane (controllo $x_0$) e vogliamo simulare come ciascuna cellula evolverà sotto l'effetto della perturbazione genetica $c$.
Dobbiamo quindi risolvere il Problema ai Valori Iniziali (IVP):
$$x(1) = x(0) + \int_0^1 v_\theta(x_t, t, c) \, dt$$

Non potendo calcolare l'integrale in forma chiusa (poiché $v_\theta$ è una rete non lineare), utilizziamo metodi di **discretizzazione numerica** suddividendo l'intervallo $[0, 1]$ in $N$ passi di ampiezza $\Delta t = 1/N$.


### 1.1 Metodo di Eulero (1° Ordine)
È l'approssimazione più semplice e intuitiva:
$$x_{t+\Delta t} = x_t + \Delta t \cdot v_\theta(x_t, t)$$
- **Vantaggi**: 1 sola valutazione della rete neurale per step (molto veloce). Con Flow Matching OT, le traiettorie sono quasi rette, quindi anche pochi step (o 1 solo step!) possono dare ottimi risultati.
- **Svantaggi**: Errore di troncamento locale $\mathcal{O}(\Delta t^2)$, errore globale $\mathcal{O}(\Delta t)$.


### 1.2 Metodo Runge-Kutta di 4° Ordine (RK4)
Campiona la velocità in 4 punti intermedi per stimare con altissima precisione la curvatura del flusso:
$$\begin{aligned}
k_1 &= v_\theta(x_t, t) \\
k_2 &= v_\theta(x_t + \frac{\Delta t}{2} k_1, t + \frac{\Delta t}{2}) \\
k_3 &= v_\theta(x_t + \frac{\Delta t}{2} k_2, t + \frac{\Delta t}{2}) \\
k_4 &= v_\theta(x_t + \Delta t \cdot k_3, t + \Delta t) \\
x_{t+\Delta t} &= x_t + \frac{\Delta t}{6}(k_1 + 2k_2 + 2k_3 + k_4)
\end{aligned}$$
- **Vantaggi**: Errore globale $\mathcal{O}(\Delta t^4)$, straordinariamente accurato anche con traiettorie curve o complesse.
- **Costo**: 4 valutazioni di rete per step temporale.


---
## 2. Setup e Import


In [ ]:
import sys, os
# Aggiungiamo la root del progetto al path
sys.path.insert(0, os.path.abspath('..'))

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

from cfm.solver import ODESolver
print('Modulo ODESolver caricato con successo!')


---
## 3. Codice di `ODESolver` Riga per Riga

Analizziamo come la classe `ODESolver` gestisce i batch di cellule, il calcolo temporale e la memorizzazione della traiettoria.


In [ ]:
import inspect
# Ispezioniamo la firma e la docstring del metodo sample
print(inspect.getsource(ODESolver.sample))


### Punti salienti dell'implementazione:
1. `@torch.no_grad()`: Durante l'integrazione numerica non dobbiamo accumulare grafi di computazione autograd, risparmiando memoria GPU/RAM.
2. `t_tensor = torch.full((batch_size, 1), t_val, ...)`: Crea un tensore temporale colonna broadcastabile per tutto il batch di cellule.
3. `return_trajectory=True`: Consente di registrare ogni stato intermedio $x_t$ per visualizzare il filmato/plot continuo dell'evoluzione cellulare.


---
## 4. Esperimento Visivo: Confronto Eulero vs RK4 su un Flusso Curvo 2D

Creiamo un campo vettoriale non lineare giocattolo con traiettorie a spirale:
$$v(x, y) = \begin{pmatrix} -y - 0.2 x \\ x - 0.2 y \end{pmatrix}$$
e confrontiamo la precisione di Eulero e RK4 al variare del numero di step.


In [ ]:
class SpiralVectorField(nn.Module):
    """Campo vettoriale giocattolo che genera una spirale convergente."""
    def forward(self, x, t, condition=None):
        # x ha forma [B, 2]
        x1, x2 = x[:, 0:1], x[:, 1:2]
        v1 = -x2 - 0.2 * x1
        v2 = x1 - 0.2 * x2
        return torch.cat([v1, v2], dim=-1)

spiral_vf = SpiralVectorField()

# Stato iniziale: una nube circolare di cellule al tempo 0
theta = torch.linspace(0, 2 * np.pi, 20)
x_0 = torch.stack([torch.cos(theta), torch.sin(theta)], dim=-1) * 2.0

# Risolviamo con Eulero con pochi step (N=6) vs RK4 con pochi step (N=6)
solver_euler_fast = ODESolver(method='euler', num_steps=6)
solver_rk4_fast = ODESolver(method='rk4', num_steps=6)
solver_rk4_ref = ODESolver(method='rk4', num_steps=100)  # Riferimento 'esatto'

_, traj_euler = solver_euler_fast.sample(spiral_vf, x_0, return_trajectory=True)
_, traj_rk4 = solver_rk4_fast.sample(spiral_vf, x_0, return_trajectory=True)
_, traj_ref = solver_rk4_ref.sample(spiral_vf, x_0, return_trajectory=True)

print('Integrazione completata con successo!')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Eulero N=6
axes[0].set_title('Eulero (N=6 passi) - Instabile/Dispersivo', fontsize=12, fontweight='bold')
traj_e = torch.stack(traj_euler).numpy()
for i in range(len(x_0)):
    axes[0].plot(traj_e[:, i, 0], traj_e[:, i, 1], 'r.-', alpha=0.6)
axes[0].scatter(x_0[:, 0], x_0[:, 1], c='blue', s=40, label='t=0 (Start)', zorder=5)
axes[0].scatter(traj_e[-1, :, 0], traj_e[-1, :, 1], c='red', s=40, label='t=1 (End)', zorder=5)
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].legend()

# 2. RK4 N=6
axes[1].set_title('RK4 (N=6 passi) - Ottima Conservazione', fontsize=12, fontweight='bold')
traj_r = torch.stack(traj_rk4).numpy()
for i in range(len(x_0)):
    axes[1].plot(traj_r[:, i, 0], traj_r[:, i, 1], 'g.-', alpha=0.6)
axes[1].scatter(x_0[:, 0], x_0[:, 1], c='blue', s=40, label='t=0 (Start)', zorder=5)
axes[1].scatter(traj_r[-1, :, 0], traj_r[-1, :, 1], c='green', s=40, label='t=1 (End)', zorder=5)
axes[1].grid(True, linestyle='--', alpha=0.5)
axes[1].legend()

# 3. RK4 N=100 (Riferimento quasi-esatto)
axes[2].set_title('RK4 (N=100 passi) - Ground Truth Continuo', fontsize=12, fontweight='bold')
traj_exact = torch.stack(traj_ref).numpy()
for i in range(len(x_0)):
    axes[2].plot(traj_exact[:, i, 0], traj_exact[:, i, 1], 'k-', alpha=0.4)
axes[2].scatter(x_0[:, 0], x_0[:, 1], c='blue', s=40, label='t=0 (Start)', zorder=5)
axes[2].scatter(traj_exact[-1, :, 0], traj_exact[-1, :, 1], c='darkviolet', s=40, label='t=1 (End)', zorder=5)
axes[2].grid(True, linestyle='--', alpha=0.5)
axes[2].legend()

plt.tight_layout()
plt.show()


### Osservazione Chiave:
- Con traiettorie a forte curvatura, **Eulero con pochi passi accumula errore numerico** e devia dal flusso reale.
- **RK4 anche con soli 6 passi segue quasi perfettamente la curva esatta**!
- Nel nostro progetto CFM-GRN usiamo di default `method='rk4'` con `num_steps=20`, garantendo massima accuratezza biochimica per il gene knockout.


---
## 5. Esercizio Interattivo

Prova a testare come si comporta `ODESolver` se passi un vettore di condizione `condition` ad una rete con perturbazione.


In [ ]:
class ConditionedVF(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Linear(3, 2)
    def forward(self, x, t, condition):
        # Concatena x [B, 2] e la perturbazione scalare [B, 1]
        inp = torch.cat([x, condition], dim=-1)
        return self.net(inp)

model_cond = ConditionedVF()
solver = ODESolver(method='rk4', num_steps=10)

x_start = torch.randn(5, 2)
cond = torch.ones(5, 1)  # Perturbazione attiva

x_final, traj = solver.sample(model_cond, x_start, condition=cond, return_trajectory=True)
print('Forma finale:', x_final.shape)
print('Lunghezza traiettoria salvata:', len(traj))
assert x_final.shape == (5, 2), 'Shape non corretta!'
print('Tutti gli assert superati!')
